# NB_bishop_ch06_figures

**Bishop Chapter 6 — Key Figures: Network Diagrams, Activations & Mixture Density Networks**

This notebook generates publication-quality figures for Bishop Chapter 6: two-layer and deep network diagrams, universal approximation illustration, activation function comparison, and mixture density network visualization.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_06/NB_bishop_ch06_figures.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

print(f'NumPy version: {np.__version__}')

In [ ]:
# Chart styling setup
COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True, dpi=300)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=300)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## Figure 6.4: Two-Layer Neural Network Diagram

A feedforward network with $D$ inputs, $M$ hidden units, and $K$ outputs.

In [ ]:
def draw_network(ax, layer_sizes, layer_labels=None, node_radius=0.25,
                 x_spacing=2.5, colors=None, title=None):
    """Draw a feedforward neural network diagram with circles and arrows."""
    if colors is None:
        colors = [COLORS['blue']] * len(layer_sizes)
    if layer_labels is None:
        layer_labels = ['' for _ in layer_sizes]

    n_layers = len(layer_sizes)
    max_nodes = max(layer_sizes)
    positions = []  # list of lists of (x, y)

    for l_idx, n_nodes in enumerate(layer_sizes):
        x = l_idx * x_spacing
        y_offset = (max_nodes - n_nodes) / 2.0
        layer_pos = []
        for n_idx in range(n_nodes):
            y = max_nodes - 1 - n_idx - y_offset
            layer_pos.append((x, y))
        positions.append(layer_pos)

    # Draw connections
    for l_idx in range(n_layers - 1):
        for (x1, y1) in positions[l_idx]:
            for (x2, y2) in positions[l_idx + 1]:
                dx = x2 - x1
                dy = y2 - y1
                dist = np.sqrt(dx**2 + dy**2)
                # Shorten by node_radius at each end
                sx = x1 + node_radius * dx / dist
                sy = y1 + node_radius * dy / dist
                ex = x2 - node_radius * dx / dist
                ey = y2 - node_radius * dy / dist
                ax.annotate('', xy=(ex, ey), xytext=(sx, sy),
                            arrowprops=dict(arrowstyle='->', color='#888888',
                                            lw=0.6, shrinkA=0, shrinkB=0))

    # Draw nodes
    for l_idx, layer_pos in enumerate(positions):
        for n_idx, (x, y) in enumerate(layer_pos):
            circle = plt.Circle((x, y), node_radius, facecolor='white',
                                edgecolor=colors[l_idx], linewidth=1.8, zorder=5)
            ax.add_patch(circle)

    # Layer labels below
    for l_idx, layer_pos in enumerate(positions):
        cx = layer_pos[0][0]
        bottom_y = min(p[1] for p in layer_pos)
        if layer_labels[l_idx]:
            ax.text(cx, bottom_y - 0.7, layer_labels[l_idx],
                    ha='center', va='top', fontsize=11, color=colors[l_idx],
                    fontweight='bold')

    ax.set_aspect('equal')
    ax.axis('off')
    total_w = (n_layers - 1) * x_spacing
    ax.set_xlim(-1, total_w + 1)
    ax.set_ylim(-1.5, max_nodes + 0.5)
    if title:
        ax.set_title(title, fontsize=13, pad=15)


# Figure 6.4: Two-layer network (D=3 inputs, M=4 hidden, K=2 outputs)
fig, ax = plt.subplots(figsize=(10, 5))
draw_network(ax, layer_sizes=[3, 4, 2],
             layer_labels=['Input layer\n($D = 3$)',
                           'Hidden layer\n($M = 4$)',
                           'Output layer\n($K = 2$)'],
             colors=[COLORS['blue'], COLORS['green'], COLORS['red']],
             title='Figure 6.4: Two-Layer Neural Network')
fig.tight_layout()
save_fig(fig, 'fig6_4_two_layer_network')
plt.show()

## Figure 6.5: Universal Approximation

Combining shifted and scaled sigmoid functions to approximate an arbitrary target function.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Target function: a bump with two peaks
x = np.linspace(-1, 6, 500)

def target_fn(x):
    return 0.6 * np.exp(-0.5 * ((x - 1.5) / 0.5)**2) + \
           0.4 * np.exp(-0.5 * ((x - 3.5) / 0.7)**2)

y_target = target_fn(x)

# Individual sigmoid "step" contributions
# Each pair of sigmoids creates a bump: w * [sigma(a*(x-c1)) - sigma(a*(x-c2))]
components = [
    {'w': 0.60, 'a': 8, 'c1': 0.8, 'c2': 2.2, 'label': '$h_1$'},
    {'w': 0.40, 'a': 6, 'c1': 2.5, 'c2': 4.5, 'label': '$h_2$'},
    {'w': 0.15, 'a': 10, 'c1': 1.0, 'c2': 1.8, 'label': '$h_3$'},
    {'w': -0.10, 'a': 12, 'c1': 3.0, 'c2': 4.0, 'label': '$h_4$'},
]

comp_curves = []
for c in components:
    curve = c['w'] * (sigmoid(c['a'] * (x - c['c1'])) - sigmoid(c['a'] * (x - c['c2'])))
    comp_curves.append(curve)

y_approx = sum(comp_curves)

comp_colors = [COLORS['blue'], COLORS['red'], COLORS['green'], COLORS['amber']]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: individual sigmoid bumps
for i, (curve, comp) in enumerate(zip(comp_curves, components)):
    axes[0].plot(x, curve, color=comp_colors[i], linewidth=1.5,
                label=comp['label'], alpha=0.8)
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('Activation')
axes[0].set_title('Individual Sigmoid Components')
axes[0].legend()

# Right: sum vs target
axes[1].plot(x, y_target, color='black', linewidth=2.5, label='Target $f(x)$', alpha=0.7)
axes[1].plot(x, y_approx, color=COLORS['red'], linewidth=2, linestyle='--',
             label='Sum of sigmoids')
axes[1].set_xlabel('$x$')
axes[1].set_ylabel('$f(x)$')
axes[1].set_title('Universal Approximation: Sum Fits Target')
axes[1].legend()

fig.suptitle('Figure 6.5: Universal Approximation via Sigmoid Combinations',
             fontsize=13, y=1.03)
fig.tight_layout()
save_fig(fig, 'fig6_5_universal_approx')
plt.show()

## Figure 6.7: Activation Functions

Comparison of ReLU, Sigmoid, Tanh, and Leaky ReLU on a single plot.

In [ ]:
z = np.linspace(-5, 5, 500)

relu = np.maximum(0, z)
leaky_relu = np.where(z > 0, z, 0.1 * z)
sig = 1.0 / (1.0 + np.exp(-z))
tanh_vals = np.tanh(z)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(z, relu, color=COLORS['blue'], linewidth=2.2, label='ReLU')
ax.plot(z, leaky_relu, color=COLORS['amber'], linewidth=2.2, label='Leaky ReLU ($\\alpha=0.1$)',
        linestyle='--')
ax.plot(z, sig, color=COLORS['red'], linewidth=2.2, label='Sigmoid')
ax.plot(z, tanh_vals, color=COLORS['green'], linewidth=2.2, label='Tanh')

ax.axhline(0, color='gray', lw=0.5, ls='-', alpha=0.4)
ax.axvline(0, color='gray', lw=0.5, ls='-', alpha=0.4)
ax.set_xlabel('$z$', fontsize=12)
ax.set_ylabel('$g(z)$', fontsize=12)
ax.set_title('Figure 6.7: Activation Functions', fontsize=13)
ax.set_ylim(-1.5, 5.2)
ax.legend()
fig.tight_layout()
save_fig(fig, 'fig6_7_relu_activation')
plt.show()

## Figure 6.9: Deep Network Diagram

A deep feedforward network with 5+ layers.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
layer_sizes = [4, 6, 6, 5, 5, 3]
layer_labels = ['Input\n($D=4$)', 'Hidden 1\n($H_1=6$)', 'Hidden 2\n($H_2=6$)',
                'Hidden 3\n($H_3=5$)', 'Hidden 4\n($H_4=5$)', 'Output\n($K=3$)']
layer_colors = [COLORS['blue'], COLORS['green'], COLORS['green'],
                COLORS['green'], COLORS['green'], COLORS['red']]

draw_network(ax, layer_sizes=layer_sizes,
             layer_labels=layer_labels,
             colors=layer_colors,
             x_spacing=2.8, node_radius=0.22,
             title='Figure 6.9: Deep Feedforward Network (6 Layers)')
fig.tight_layout()
save_fig(fig, 'fig6_9_deep_network')
plt.show()

## Figure 6.15: Mixture Density Network

An inverse problem where $x = \sin(t) + \varepsilon$, $y = t$ creates a multi-valued mapping. A single-output network fails; a mixture density network captures all modes.

In [ ]:
# Generate multi-valued data
np.random.seed(42)
N = 1500
t = np.random.uniform(0, 2 * np.pi, N)
noise = np.random.normal(0, 0.05, N)
x_data = np.sin(t) + noise
y_data = t

# "Single output" fit: just the conditional mean
# For visualization, bin x and compute mean y per bin
x_sorted_idx = np.argsort(x_data)
x_sorted = x_data[x_sorted_idx]
y_sorted = y_data[x_sorted_idx]

# Moving average as proxy for single-output prediction
window = 60
x_smooth = np.convolve(x_sorted, np.ones(window)/window, mode='valid')
y_smooth = np.convolve(y_sorted, np.ones(window)/window, mode='valid')

# Simulate mixture density output:
# For each x value, the true conditional has multiple modes
# We plot the arcsin branches as the "mixture means"
x_grid = np.linspace(-0.95, 0.95, 200)
# Branches of arcsin
branch1 = np.arcsin(x_grid)                      # [−pi/2, pi/2] shifted to [0, pi]
branch2 = np.pi - np.arcsin(x_grid)              # [pi/2, 3pi/2]
branch3 = 2 * np.pi + np.arcsin(x_grid)          # [3pi/2, 5pi/2]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: Data
axes[0].scatter(x_data, y_data, s=2, alpha=0.3, color=COLORS['blue'])
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('$t$')
axes[0].set_title('Data: $x = \\sin(t) + \\varepsilon$')

# Panel 2: Single-output failure
axes[1].scatter(x_data, y_data, s=2, alpha=0.15, color='gray')
axes[1].plot(x_smooth, y_smooth, color=COLORS['red'], linewidth=2.5,
             label='Single output (mean)')
axes[1].set_xlabel('$x$')
axes[1].set_ylabel('$t$')
axes[1].set_title('Single Output Fails')
axes[1].legend()

# Panel 3: MDN branches
axes[2].scatter(x_data, y_data, s=2, alpha=0.15, color='gray')
axes[2].plot(x_grid, branch1, color=COLORS['blue'], linewidth=2, label='$\\mu_1$')
axes[2].plot(x_grid, branch2, color=COLORS['red'], linewidth=2, label='$\\mu_2$')
axes[2].plot(x_grid, branch3, color=COLORS['green'], linewidth=2, label='$\\mu_3$')
axes[2].set_xlabel('$x$')
axes[2].set_ylabel('$t$')
axes[2].set_title('MDN: Mixture Component Means')
axes[2].legend()

fig.suptitle('Figure 6.15: Mixture Density Network for Multi-Valued Mappings',
             fontsize=13, y=1.03)
fig.tight_layout()
save_fig(fig, 'fig6_15_mixture_density')
plt.show()

In [ ]:
print('Notebook complete.')